In [ ]:
!pip install "rfdetr[train]>=1.8.3" pandas --quiet

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
from pathlib import Path
DATASET_DIR = Path('/content/dataset')
required_annotations = [DATASET_DIR / split / '_annotations.coco.json' for split in ('train', 'valid', 'test')]
missing = [path for path in required_annotations if not path.exists()]
if missing:
    missing_paths = '\n'.join((f'- {path}' for path in missing))
    raise FileNotFoundError('Uma exportação COCO é obrigatória. Arquivos de anotação ausentes:\n' + missing_paths)
print(f'Using COCO dataset: {DATASET_DIR.resolve()}')

In [ ]:
from rfdetr import RFDETRMedium, RFDETRNano
VARIANT = 'medium'
NUM_CLASSES = 7
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 0.0001
RESOLUTION = {'nano': 384, 'medium': 576}[VARIANT]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('runs') / f'rfdetr-{VARIANT}-urban-disaster'
RUN_TEST = True
model_class = {'nano': RFDETRNano, 'medium': RFDETRMedium}[VARIANT]
model = model_class(num_classes=NUM_CLASSES)
model.train(dataset_dir=str(DATASET_DIR), output_dir=str(OUTPUT_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS, lr=LEARNING_RATE, resolution=RESOLUTION, device=DEVICE, eval_interval=1, run_test=RUN_TEST)

In [ ]:
import pandas as pd
metrics_path = OUTPUT_DIR / 'metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    metric_columns = [column for column in ('epoch', 'val/mAP_50', 'val/mAP_50_95', 'val/precision', 'val/recall', 'val/F1') if column in metrics.columns]
    display(metrics[metric_columns].dropna(how='all').tail())
else:
    print(f'Execute o treinamento primeiro; métricas não encontradas em {metrics_path}')
for artifact in ('training_config.json', 'results.json', 'checkpoint_best_total.pth'):
    print(f'{artifact}: {(OUTPUT_DIR / artifact).exists()}')

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
test_images = sorted((path for path in (DATASET_DIR / 'test').iterdir() if path.suffix.lower() in image_extensions))
if not test_images:
    raise FileNotFoundError('Nenhuma imagem de teste foi encontrada ao lado de test/_annotations.coco.json.')
checkpoint_path = OUTPUT_DIR / 'checkpoint_best_total.pth'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Melhor checkpoint não encontrado: {checkpoint_path}')
best_model = model_class(num_classes=NUM_CLASSES, pretrain_weights=str(checkpoint_path))
detections = best_model.predict(str(test_images[0]), threshold=0.3)
print(f'Imagem de teste: {test_images[0].name}')
print(f'Detecções: {len(detections.xyxy)}')
detections